# 10 Vector Autoregression

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/06-Econometrics/10_Vector_Autoregression.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=06-Econometrics/10_Vector_Autoregression.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import Markdown, display
from statsmodels.tsa.api import VAR

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (12, 8), 'figure.dpi': 150,
                     'axes.titlesize': 'large', 'axes.labelsize': 'medium',
                     'xtick.labelsize': 'small', 'ytick.labelsize': 'small'})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)

# --- Utility Functions ---


## Table of Contents

1. [Introduction](#Introduction)


## The Lens: Everything Depends on Everything
**What problem are we solving?**
Macroeconomic variables are endogenous. GDP affects Interest Rates, and Interest Rates affect GDP. Single-equation models cannot capture this feedback loop.

**Vector Autoregression (VAR)** models treat all variables as endogenous. It is a system of equations where every variable depends on the past values of every variable.

**Why this method?**
VARs allow us to analyze dynamics without imposing strong theoretical restrictions. They are the workhorse tool for empirical macroeconomics, enabling Granger causality tests, impulse response analysis, and forecast error variance decompositions.



**Economic question.** In *10 Vector Autoregression*, what must remain economically invariant when the computational representation changes? The computational task only has economic meaning after the estimand and identifying assumptions are explicit. Ask what variation identifies the parameter, which observations act as the comparison group, and what data-generating process would make the estimator fail. A good empirical workflow pairs the point estimate with diagnostics, uncertainty, and at least one falsification or sensitivity check so that precision is not confused with identification.

### Learning Objectives
* **Specify** a VAR(p) model and select the lag order using AIC/BIC.
* **Conduct** Granger causality tests and interpret temporal precedence.
* **Compute** orthogonalized impulse response functions using the Cholesky decomposition.
* **Perform** forecast error variance decomposition to assess shock contributions.

### Prerequisites
* **Time Series:** ARMA models, stationarity, and the Box-Jenkins workflow (Module 06 or 08).
* **Linear Algebra:** Matrix operations and eigenvalue stability conditions.
* **Python:** `statsmodels` VAR API and Pandas DataFrames.
* **Learning-path prerequisite:** [`09_Classical_Time_Series_Analysis.ipynb`](09_Classical_Time_Series_Analysis.ipynb)


> **Learning path:** Building on [`09_Classical_Time_Series_Analysis.ipynb`](09_Classical_Time_Series_Analysis.ipynb); next continue with [`11_Bayesian_Econometrics.ipynb`](11_Bayesian_Econometrics.ipynb).


### 1. The VAR(p) Model

A VAR(p) model expresses a vector of $k$ endogenous variables, $y_t$, as a linear function of $p$ of its own lags and a vector of innovations (shocks), $u_t$.

For a simple VAR(1) with two variables ($y_{1,t}$ and $y_{2,t}$), the system is:
$$ y_{1,t} = c_1 + \phi_{11,1} y_{1,t-1} + \phi_{12,1} y_{2,t-1} + u_{1,t} $$ 
$$ y_{2,t} = c_2 + \phi_{21,1} y_{1,t-1} + \phi_{22,1} y_{2,t-1} + u_{2,t} $$

In matrix form, a general VAR(p) is written as:
$$ y_t = c + \Phi_1 y_{t-1} + \Phi_2 y_{t-2} + ... + \Phi_p y_{t-p} + u_t $$

where $y_t$ is a ($k \times 1$) vector, $c$ is a ($k \times 1$) vector of intercepts, $\Phi_i$ are ($k \times k$) coefficient matrices, and $u_t$ is a ($k \times 1$) vector of innovations with covariance matrix $E[u_t u_t'] = \Sigma$. The key assumption is that the innovations are serially uncorrelated but may be contemporaneously correlated (i.e., $\Sigma$ is not necessarily diagonal).


### 2. Estimation

Because each equation in the VAR system has the same set of regressors (the lagged values of all variables), the system can be estimated efficiently and consistently by applying Ordinary Least Squares (OLS) to each equation individually. This simplifies the estimation process significantly.


### 3. Structural VARs and Identification

The estimated innovations, $u_t$, are called **reduced-form shocks**. They are the one-step-ahead forecast errors from the model, but they are not economically meaningful as "structural" shocks (e.g., a pure monetary policy shock or technology shock). This is because they are contemporaneously correlated ($\Sigma$ is not diagonal). For example, a surprise increase in the federal funds rate ($u_{FFR,t}$) might be correlated with a simultaneous change in expected inflation ($u_{INF,t}$), so it isn't a clean policy shock.

To identify structural shocks, $\epsilon_t$, which are by definition orthogonal, we need to impose restrictions. The relationship between the reduced-form and structural shocks is:
$$ A u_t = B \epsilon_t \implies u_t = A^{-1} B \epsilon_t $$  
The goal is to find the matrices $A$ and $B$. A common and simple identification scheme is the **Cholesky decomposition**. It imposes a recursive ordering on the variables. The first variable is assumed to be affected only by its own structural shock contemporaneously. The second variable is affected by its own shock and the first shock, and so on. This corresponds to choosing $B=I$ and making $A$ a lower triangular matrix. This is achieved by finding the Cholesky factor of the reduced-form covariance matrix $\Sigma$.


### 4. Code Example: A Monetary Policy VAR

We will estimate a simple VAR for the U.S. economy using quarterly data on GDP growth, inflation, and the federal funds rate. We will then trace out the effects of a monetary policy shock (an unexpected increase in the federal funds rate) using IRFs.


### Estimating a Monetary Policy VAR


In [ ]:

# Load classic macro data from statsmodels
data = sm.datasets.macrodata.load_pandas().data
data['year'] = data['year'].astype(int)
data.index = pd.to_datetime(data['year'].astype(str) + 'Q' + data['quarter'].astype(str))

# Prepare the data: GDP growth, inflation (from CPI), and the fed funds rate
df = pd.DataFrame({
    'gdp_growth': 100 * data['realgdp'].pct_change(),
    'inflation': 100 * data['cpi'].pct_change(),
    'fed_funds': data['tbilrate'] # Using t-bill rate as a proxy
}).dropna()

# --- Estimate the VAR model ---
model = VAR(df)
results = model.fit(maxlags=15, ic='aic') # Use AIC to select the optimal lag order


In [ ]:
display(Markdown(f"> **Note:** Optimal lag order chosen by AIC: {results.k_ar}"))


In [ ]:

# --- Generate and Plot Impulse Responses ---


> **Note:** Impulse responses to a 1 S.D. shock to the Federal Funds Rate:


In [ ]:
irf = results.irf(periods=20)
fig = irf.plot(orth=True, signif=0.05) # Orth=True applies the Cholesky decomposition
fig.suptitle('Impulse Responses to a Monetary Policy Shock (Cholesky)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


> **Note:** **Interpretation:** The results show a classic monetary policy trade-off. An unexpected 1 S.D. increase in the federal funds rate (a contractionary shock) leads to a statistically significant fall in inflation and a temporary, significant decline in GDP growth. This illustrates the power of VARs to uncover dynamic relationships in the data without imposing strong theoretical assumptions.


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$y_{1,t} = c_1 + \phi_{11,1} y_{1,t-1} + \phi_{12,1} y_{2,t-1} + u_{1,t}$$

**2. Core relation**

$$y_{2,t} = c_2 + \phi_{21,1} y_{1,t-1} + \phi_{22,1} y_{2,t-1} + u_{2,t}$$

**3. Core relation**

$$y_t = c + \Phi_1 y_{t-1} + \Phi_2 y_{t-2} + ... + \Phi_p y_{t-p} + u_t$$

**4. Core relation**

$$A u_t = B \epsilon_t \implies u_t = A^{-1} B \epsilon_t$$


### Three-Tier Practice Ladder

**1. Mechanism and assumptions (Conceptual):** Define the estimand in **10 Vector Autoregression**, list the identifying assumptions, and give a concrete data-generating process that violates one assumption while leaving the others intact.

**2. Reproduce and diagnose (Applied):** Implement or reproduce the estimator using the material on 1. The VAR(p) Model, 2. Estimation. Report uncertainty and at least two diagnostics; then compare with an alternative specification that targets the same estimand.

**3. Robust extension (Challenge):** Run a Monte Carlo or sensitivity exercise that varies the most fragile identifying condition. Quantify bias/coverage or the range of estimates and state what evidence would change your substantive conclusion.

> Use the existing exercises above when they target the same skill; this ladder makes the intended progression explicit rather than replacing instructor-authored problems.


## 5. Exercises

1.  **Alternative Ordering:** The Cholesky decomposition is sensitive to the ordering of the variables. Re-estimate the VAR, but change the order of the variables in the DataFrame to `['inflation', 'gdp_growth', 'fed_funds']`. How do the impulse responses to a federal funds rate shock change? Why? This highlights the importance of the identifying assumptions.

2.  **Forecast Error Variance Decomposition (FEVD):** The FEVD shows the proportion of the forecast error variance for each variable that is attributable to its own shocks versus shocks to other variables. Use the `results.fevd()` method to compute and plot the FEVD for a 20-quarter horizon. What does it tell you about the relative importance of monetary policy shocks for explaining fluctuations in GDP growth and inflation?

3.  **Historical Decomposition:** The `results.plot_acorr()` method can be used to plot the historical decomposition of the variables, showing how the different structural shocks have contributed to their evolution over time. Generate and interpret this plot.

4.  **Granger Causality:** Use the `results.test_causality()` method to test whether the federal funds rate "Granger-causes" GDP growth. What is the null hypothesis of this test, and what does the result imply?


# Summary

VARs provide a flexible framework for analyzing multivariate time series dynamics.

**Key Takeaways:**
*   **Identification:** Reduced-form residuals are correlations. To talk about causation (shocks), we need identification assumptions (e.g., Cholesky ordering).
*   **Ordering Matters:** In a Cholesky decomposition, the order of variables determines the causal chain of contemporaneous shocks.
*   **Granger Causality:** A statistical test of predictive power, not true physical causality.


## References & Further Reading

- Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data* (2nd ed.). MIT Press.
- Angrist, J. D. & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
- Imbens, G. W. & Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences*. Cambridge University Press.
